# LauzHack Analysis Dataset Prep

Build one analysis-ready projects dataframe by combining yearly project files with yearly hackathon metadata.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

DATA_ROOT = Path('path here')
PROVIDER_PREFIX = 'lauzhack'


In [7]:
def _load_hackathon_metadata_row(metadata_json_path: Path, year: int) -> dict:
    metadata = json.loads(metadata_json_path.read_text(encoding='utf-8'))
    if isinstance(metadata, list):
        metadata = metadata[0] if metadata else {}
    if not isinstance(metadata, dict):
        metadata = {}

    row = {f'hackathon_{k}': v for k, v in metadata.items()}
    row['hackathon_year'] = year
    return row


def load_projects_analysis_ready(data_root: Path, provider_prefix: str = 'lauzhack') -> pd.DataFrame:
    frames: list[pd.DataFrame] = []

    for folder in sorted(data_root.glob(f'{provider_prefix}-*')):
        if not folder.is_dir():
            continue

        year_str = folder.name.split('-')[-1]
        if not year_str.isdigit():
            continue
        year = int(year_str)

        projects_path = folder / f'{provider_prefix}_projects.parquet'
        github_projects_path = folder / f'{provider_prefix}_github_project_metadata.parquet'
        metadata_json_path = folder / f'{provider_prefix}_metadata.json'

        if not projects_path.exists() or not metadata_json_path.exists():
            continue

        # Prefer project-level GitHub-enriched file when present.
        source_path = github_projects_path if github_projects_path.exists() else projects_path
        df = pd.read_parquet(source_path).copy()
        df['year'] = year

        metadata_row = _load_hackathon_metadata_row(metadata_json_path, year)
        for col, val in metadata_row.items():
            df[col] = val

        frames.append(df)

    if not frames:
        return pd.DataFrame()

    merged = pd.concat(frames, ignore_index=True)

    # Harmonize expected GitHub columns even when some years have no github_project_metadata file.
    if 'github_repo_urls' not in merged.columns:
        merged['github_repo_urls'] = [[] for _ in range(len(merged))]
    if 'github_repos_metadata' not in merged.columns:
        merged['github_repos_metadata'] = [[] for _ in range(len(merged))]
    if 'github_repo_count' not in merged.columns:
        merged['github_repo_count'] = 0

    merged['github_repo_count'] = pd.to_numeric(merged['github_repo_count'], errors='coerce').fillna(0).astype(int)

    return merged


In [8]:
projects_analysis_ready = load_projects_analysis_ready(DATA_ROOT, PROVIDER_PREFIX)

print('rows:', len(projects_analysis_ready))
print('columns:', len(projects_analysis_ready.columns))
print('years:', sorted(projects_analysis_ready['year'].dropna().unique().tolist()))

projects_analysis_ready.head()


rows: 213
columns: 21
years: [2023, 2024, 2025]


,id,title,description,url,team,awards,categories,hackathon_name,hackathon_year,hackathon_location,...,project_id,project_title,github_repo_urls,github_repo_count,github_repos_metadata,year,hackathon_source_url,hackathon_description,hackathon_social_links,hackathon_extracted_at
0,1,"SightSync, a virtual assistant for visual impa...",An app that describes the surroundings through...,https://github.com/sightsync,"[""Oriol Agost Batalla"", ""Ferran Aran""]","[""1st place overall""]","[""1st place overall""]",LauzHack 2023,2023,"EPFL, Lausanne, Switzerland",...,1,"SightSync, a virtual assistant for visual impa...","[""https://github.com/sightsync/.github"", ""http...",4,"[{""input_url"": ""https://github.com/sightsync/....",2023,https://2023.lauzhack.com,"Student-run hackathon at EPFL, Switzerland",NaN,2026-02-27T09:45:55.882193
1,2,VirtuWheel,Real city driving simulator with hand pose rec...,https://github.com/alvaro-budria/VirtuWheel,"[""Jaume Ros Alonso""]","[""2nd place overall""]","[""2nd place overall""]",LauzHack 2023,2023,"EPFL, Lausanne, Switzerland",...,2,VirtuWheel,"[""https://github.com/alvaro-budria/VirtuWheel""]",1,"[{""input_url"": ""https://github.com/alvaro-budr...",2023,https://2023.lauzhack.com,"Student-run hackathon at EPFL, Switzerland",NaN,2026-02-27T09:45:55.882193
2,3,It’s not about winning,"""It’s not about winning"" is a cutting-edge app...",https://fastmaildassdas.retool.com/apps/92b217...,"[""Indira Fömmel"", ""Leonard Eyer"", ""Gero Embser""]","[""3rd place overall"", ""AXA challenge winner""]","[""3rd place overall"", ""AXA challenge winner""]",LauzHack 2023,2023,"EPFL, Lausanne, Switzerland",...,3,It’s not about winning,[],0,[],2023,https://2023.lauzhack.com,"Student-run hackathon at EPFL, Switzerland",NaN,2026-02-27T09:45:55.882193
3,4,BMS detction HCM,model learning and qualtitative feedback diagn...,https://github.com/Gabriel29062001/hackathon,"[""Eddy BESSAH"", ""gabriel paffi"", ""Grégoire Lon...","[""Organizers' prize"", ""BMS challenge winner""]","[""Organizers' prize"", ""BMS challenge winner""]",LauzHack 2023,2023,"EPFL, Lausanne, Switzerland",...,4,BMS detction HCM,"[""https://github.com/Gabriel29062001/hackathon""]",1,"[{""input_url"": ""https://github.com/Gabriel2906...",2023,https://2023.lauzhack.com,"Student-run hackathon at EPFL, Switzerland",NaN,2026-02-27T09:45:55.882193
4,5,AWS Challenge,Our take on obtaining useful and compact infor...,https://github.com/mgil4/AWS_LAUZ,"[""Álvaro Domingo Reguero"", ""Gustavo Vergara Ga...","[""AWS challenge winner""]","[""AWS challenge winner""]",LauzHack 2023,2023,"EPFL, Lausanne, Switzerland",...,5,AWS Challenge,"[""https://github.com/mgil4/AWS_LAUZ""]",1,"[{""input_url"": ""https://github.com/mgil4/AWS_L...",2023,https://2023.lauzhack.com,"Student-run hackathon at EPFL, Switzerland",NaN,2026-02-27T09:45:55.882193


In [9]:
# Optional export for downstream analysis scripts
out_csv = DATA_ROOT / 'all_lauzhack_projects_analysis_ready.csv'
out_parquet = DATA_ROOT / 'all_lauzhack_projects_analysis_ready.parquet'

projects_analysis_ready.to_csv(out_csv, index=False)

def _jsonable(v):
    if v is None:
        return None
    if isinstance(v, dict):
        return {str(k): _jsonable(val) for k, val in v.items()}
    if isinstance(v, (list, tuple, set)):
        return [_jsonable(val) for val in v]
    if hasattr(v, 'tolist') and not isinstance(v, (str, bytes, bytearray)):
        return _jsonable(v.tolist())
    if hasattr(v, 'item') and not isinstance(v, (str, bytes, bytearray)):
        item_v = v.item()
        if item_v is not v:
            return _jsonable(item_v)
    if hasattr(v, 'isoformat') and not isinstance(v, (str, bytes, bytearray)):
        return v.isoformat()
    return v

# Parquet-safe copy: serialize object columns to JSON strings after normalization.
parquet_ready = projects_analysis_ready.copy()
for col in parquet_ready.columns:
    if parquet_ready[col].dtype == 'object':
        parquet_ready[col] = parquet_ready[col].map(_jsonable)
        if parquet_ready[col].map(lambda v: isinstance(v, (list, dict))).any():
            parquet_ready[col] = parquet_ready[col].map(
                lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v
            )

parquet_ready.to_parquet(out_parquet, index=False)

print('wrote:', out_csv)
print('wrote:', out_parquet)


wrote: /Users/eisha/Documents/hackathon_analysis/data/all_lauzhack_projects_analysis_ready.csv
wrote: /Users/eisha/Documents/hackathon_analysis/data/all_lauzhack_projects_analysis_ready.parquet


In [10]:
# Quick readiness checks
projects_analysis_ready.groupby('year').agg(
    projects=('title', 'count'),
    projects_with_repos=('github_repo_count', lambda s: int((s > 0).sum())),
    total_linked_repos=('github_repo_count', 'sum'),
)


,projects,projects_with_repos,total_linked_repos
year,,,
2023,67,58,63
2024,79,75,85
2025,67,64,68
